In [0]:

# Create widgets (ordered: source_system → table_name → landing_timestamp)
dbutils.widgets.text("source_system", "", "1. Source System")
dbutils.widgets.text("table_name", "", "2. Table Name")
dbutils.widgets.text("landing_timestamp", "", "3. Landing Timestamp")
dbutils.widgets.text("source_file_format", "", "4. Source File Format")




In [0]:
# Get widget values
source_system = dbutils.widgets.get("source_system")
table_name = dbutils.widgets.get("table_name")
landing_timestamp = dbutils.widgets.get("landing_timestamp")
source_file_format = dbutils.widgets.get("source_file_format")

In [0]:
import json

landing_path = f'/Volumes/adb_caresync/landing/landing_volume/{source_system}/{table_name}/{landing_timestamp}/'

# If no new file has arrived (folder missing or empty), end gracefully so the ADF activity succeeds
try:
    landing_files = [f for f in dbutils.fs.ls(landing_path) if not f.name.startswith(('_', '.'))]
except Exception as e:
    if 'FileNotFound' in str(e) or 'No such file or directory' in str(e):
        landing_files = []
    else:
        raise

if not landing_files:
    print(f"No new files found at {landing_path}. Skipping load.")
    dbutils.notebook.exit(json.dumps({"status": "NO_NEW_FILE", "record_count": 0, "landing_path": landing_path}))

if source_file_format == 'csv':
    landing_df = (spark.read
        .option("header", "true")
        .option("inferSchema", "true")
        .csv(landing_path)
    )
elif source_file_format == 'parquet':
    landing_df = spark.read.parquet(landing_path)
elif source_file_format == 'json':
    landing_df = spark.read.option("multiLine", "true").json(landing_path)
else:
    raise ValueError(f"Unsupported source file format: '{source_file_format}'. Supported formats: csv, parquet, json.")

print(f"Successfully read {landing_df.count()} rows from {landing_path} in '{source_file_format}' format.")
landing_df.display()


In [0]:
from pyspark.sql.functions import lit, current_timestamp

updated_df = landing_df.withColumn("landing_timestamp", lit(landing_timestamp)) \
                       .withColumn("insert_timestamp", current_timestamp())

In [0]:
display(updated_df)

In [0]:
updated_df.write.mode("append").saveAsTable(f"adb_caresync.bronze.{table_name}")